## Task 1 — Multi-Agent Design Thinking

This project uses a realistic business workflow: research a competitor, analyze the findings, and create a stakeholder-ready marketing recommendation.
The work is divided among three specialized agents so that research, analysis, and communication are handled by separate responsibilities.



In [5]:
# Task 1 — Multi-Agent Design Thinking

business_task = """
Research a competing technology company, analyze its positioning and offerings,
identify useful marketing insights, and prepare a stakeholder-ready marketing recommendation.
"""
agent_1_design = {
    "role": "Competitor Research Specialist",
    "goal": "Collect relevant and reliable information about the competitor, including products, positioning, target customers, and differentiators.",
    "backstory": "You are an experienced market research specialist. You focus on collecting relevant evidence and separating useful facts from assumptions."
}

agent_2_design = {
    "role": "Business Insights Analyst",
    "goal": "Analyze competitor research and identify strengths, weaknesses, opportunities, threats, and actionable business insights.",
    "backstory": "You are a business analyst experienced in turning research findings into structured insights that help teams make better decisions."
}

agent_3_design = {
    "role": "Marketing Strategy Specialist",
    "goal": "Use the research and business insights to create a practical marketing angle suitable for stakeholders.",
    "backstory": "You are a marketing strategist who specializes in converting business research into clear positioning and actionable recommendations."
}

print("Business task:")
print(business_task)

print("\nAgent 1:")
print("Role:", agent_1_design["role"])
print("Goal:", agent_1_design["goal"])
print("Backstory:", agent_1_design["backstory"])

print("\nAgent 2:")
print("Role:", agent_2_design["role"])
print("Goal:", agent_2_design["goal"])
print("Backstory:", agent_2_design["backstory"])

print("\nAgent 3:")
print("Role:", agent_3_design["role"])
print("Goal:", agent_3_design["goal"])
print("Backstory:", agent_3_design["backstory"])

print("\nWhy multiple specialized agents?")
print(
    "Specialized agents can outperform one generalist because each agent focuses "
    "on a different stage of the workflow and can produce more focused work."
)

print(
    "However, a single agent may be better for small, simple tasks where delegation "
    "adds unnecessary latency, cost, and coordination complexity."
)

Business task:

Research a competing technology company, analyze its positioning and offerings,
identify useful marketing insights, and prepare a stakeholder-ready marketing recommendation.


Agent 1:
Role: Competitor Research Specialist
Goal: Collect relevant and reliable information about the competitor, including products, positioning, target customers, and differentiators.
Backstory: You are an experienced market research specialist. You focus on collecting relevant evidence and separating useful facts from assumptions.

Agent 2:
Role: Business Insights Analyst
Goal: Analyze competitor research and identify strengths, weaknesses, opportunities, threats, and actionable business insights.
Backstory: You are a business analyst experienced in turning research findings into structured insights that help teams make better decisions.

Agent 3:
Role: Marketing Strategy Specialist
Goal: Use the research and business insights to create a practical marketing angle suitable for stakeholders.
B

## Task 2 — Build Agents & Assign Tools

Each agent receives a role-specific LLM configuration and only the tools needed for its responsibility.
The researcher receives search access, while the analyst and marketing specialist work primarily from the outputs supplied by earlier agents.


In [38]:
%pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 18.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
bigframes 2.48.0 requires rich<14,>=12.4.4, but you have rich 14.3.4 which is incompatible.


In [49]:
import os
from google.colab import userdata

os.environ.pop("OPENAI_API_KEY", None)
os.environ.pop("GOOGLE_API_KEY", None)

api_key = userdata.get("GEMINI_API_KEY").strip()

print("Gemini key loaded:", bool(api_key))
print("Key length:", len(api_key))

Gemini key loaded: True
Key length: 14


In [51]:
print("Key loaded:", bool(api_key))
print("Key length:", len(api_key))
print("Starts with:", api_key[:4])
print("Ends with:", api_key[-4:])

Key loaded: True
Key length: 14
Starts with: GEMI
Ends with: _KEY


In [ ]:
%pip install -U crewai crewai-tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00


In [40]:
# Task 2 — Build Agents & Assign Tools

import os
from crewai import Agent, LLM
from crewai_tools import SerperDevTool

print("Task 2 — Build Agents & Assign Tools")
print("CrewAI agent configuration started.")

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY is not set. Please set your OpenAI API key before running this cell."
    )

if not os.getenv("SERPER_API_KEY"):
    raise EnvironmentError(
        "SERPER_API_KEY is not set. Please set your Serper API key before running this cell."
    )

research_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.2
)

analysis_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.2
)

marketing_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.4
)

search_tool = SerperDevTool()

research_agent = Agent(
    role="Competitor Research Specialist",
    goal=(
        "Research the competitor using reliable web information and produce "
        "structured evidence about products, positioning, customers, and differentiators."
    ),
    backstory=(
        "You are an experienced market research specialist. "
        "You focus on collecting factual information, identifying useful evidence, "
        "and clearly separating facts from assumptions."
    ),
    llm=research_llm,
    tools=[search_tool],
    verbose=True,
    allow_delegation=False
)

analysis_agent = Agent(
    role="Business Insights Analyst",
    goal=(
        "Analyze the competitor research and identify strengths, weaknesses, "
        "opportunities, threats, and actionable business insights."
    ),
    backstory=(
        "You are an experienced business analyst who specializes in transforming "
        "research findings into structured insights that support business decisions."
    ),
    llm=analysis_llm,
    tools=[],
    verbose=True,
    allow_delegation=False
)

marketing_agent = Agent(
    role="Marketing Strategy Specialist",
    goal=(
        "Turn the research and business analysis into a clear, realistic, "
        "and stakeholder-ready marketing recommendation."
    ),
    backstory=(
        "You are an experienced technology marketing strategist. "
        "You specialize in converting business research into practical positioning, "
        "messaging, and marketing actions."
    ),
    llm=marketing_llm,
    tools=[],
    verbose=True,
    allow_delegation=False
)

print("\nAgents created successfully.")

print("\nAgent 1")
print("Role:", research_agent.role)
print("Goal:", research_agent.goal)
print("Tools:", [type(tool).__name__ for tool in research_agent.tools])

print("\nAgent 2")
print("Role:", analysis_agent.role)
print("Goal:", analysis_agent.goal)
print("Tools:", [type(tool).__name__ for tool in analysis_agent.tools])

print("\nAgent 3")
print("Role:", marketing_agent.role)
print("Goal:", marketing_agent.goal)
print("Tools:", [type(tool).__name__ for tool in marketing_agent.tools])

print("\nTool assignment justification:")
print(
    "The Competitor Research Specialist receives SerperDevTool because "
    "web search is required to collect current competitor information."
)
print(
    "The Business Insights Analyst receives no external tool because its "
    "main input is the research output produced by the first agent."
)
print(
    "The Marketing Strategy Specialist receives no external tool because "
    "it transforms the research and analysis into a stakeholder-ready recommendation."
)

Task 2 — Build Agents & Assign Tools
CrewAI agent configuration started.

Agents created successfully.

Agent 1
Role: Competitor Research Specialist
Goal: Research the competitor using reliable web information and produce structured evidence about products, positioning, customers, and differentiators.
Tools: ['SerperDevTool']

Agent 2
Role: Business Insights Analyst
Goal: Analyze the competitor research and identify strengths, weaknesses, opportunities, threats, and actionable business insights.
Tools: []

Agent 3
Role: Marketing Strategy Specialist
Goal: Turn the research and business analysis into a clear, realistic, and stakeholder-ready marketing recommendation.
Tools: []

Tool assignment justification:
The Competitor Research Specialist receives SerperDevTool because web search is required to collect current competitor information.
The Business Insights Analyst receives no external tool because its main input is the research output produced by the first agent.
The Marketing Strate

## Task 3 — Define Tasks & Process

The three CrewAI tasks form a sequential pipeline: research is completed first, analysis uses the research output, and marketing strategy uses both previous outputs.
The crew is executed with Process.sequential so each stage receives the real output generated by the previous stage.

In [ ]:
from crewai import Task, Crew, Process

print("Task 3 — Define Tasks & Process")

research_task = Task(
    description="""
Research Microsoft as a technology competitor.

Collect information about:
1. Major products or services
2. Target customers
3. Market positioning
4. Important differentiators
5. Recent strategic or marketing themes when reliable information is available

Use the assigned search tool.
Do not invent facts.

Return:
COMPETITOR:
PRODUCTS_AND_SERVICES:
TARGET_CUSTOMERS:
POSITIONING:
DIFFERENTIATORS:
EVIDENCE:
UNCERTAINTIES:
""",
    expected_output="""
A structured competitor research report covering the competitor,
products/services, target customers, positioning, differentiators,
evidence, and uncertainties.
""",
    agent=research_agent
)

analysis_task = Task(
    description="""
Analyze the competitor research from the previous task.

Identify:
1. Strongest competitive advantages
2. Important customer segments
3. Marketing opportunities
4. Marketing threats
5. Three actionable business insights

Clearly separate facts from interpretation.
""",
    expected_output="""
A business analysis containing competitive advantages, customer insights,
marketing opportunities, marketing threats, and three actionable insights.
""",
    agent=analysis_agent,
    context=[research_task]
)

marketing_task = Task(
    description="""
Create a stakeholder-ready marketing recommendation using the competitor
research and business analysis.

Include:
1. Marketing position
2. Target customer
3. Competitive message
4. Three concrete marketing actions
5. Risks and limitations
6. Implementation priority
7. Final recommendation
""",
    expected_output="""
A stakeholder-ready marketing recommendation with a clear position,
target customer, competitive message, three actions, risks,
priority, and final recommendation.
""",
    agent=marketing_agent,
    context=[research_task, analysis_task]
)

sequential_crew = Crew(
    agents=[
        research_agent,
        analysis_agent,
        marketing_agent
    ],
    tasks=[
        research_task,
        analysis_task,
        marketing_task
    ],
    process=Process.sequential,
    verbose=True
)

print("Sequential Crew created.")
print("Process: sequential")
print("Number of agents:", len(sequential_crew.agents))
print("Number of tasks:", len(sequential_crew.tasks))

print("\nStarting REAL CrewAI execution...\n")

sequential_result = await sequential_crew.kickoff_async()

print("\nREAL SEQUENTIAL CREW RESULT")
print("=" * 60)
print(sequential_result)

## Task 4 — Hierarchical Delegation

The hierarchical version introduces a manager agent that coordinates the specialist agents instead of relying only on a fixed sequential flow.
The same business objective is used so the sequential and hierarchical approaches can be compared fairly.

In [ ]:
# Task 4 — Hierarchical Delegation

from crewai import Agent, Task, Crew, Process, LLM

# competitor_name is used below in hierarchical_research_task's f-string.
# It was referenced but never defined in the original notebook, which caused
# a NameError. Kept consistent with the competitor used in Task 3 (Microsoft).
competitor_name = "Microsoft"

manager_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.2
)

manager_agent = Agent(
    role="Crew Manager",
    goal=(
        "Coordinate the specialist agents, delegate work appropriately, "
        "review their results, identify missing information, and produce a reliable "
        "final business recommendation."
    ),
    backstory=(
        "You are an experienced project manager who supervises research, "
        "business analysis, and marketing strategy specialists. "
        "You delegate work carefully and review outputs before accepting the final result."
    ),
    llm=manager_llm,
    verbose=True,
    allow_delegation=True
)

hierarchical_research_task = Task(
    description=f"""
    Research {competitor_name} for the following business objective:

    {business_task}

    Gather reliable information about:
    - Products and services
    - Target customers
    - Positioning
    - Differentiators
    - Relevant recent strategic or marketing themes

    Clearly identify evidence and uncertainty.
    Do not invent facts.
    """,
    expected_output="""
    A factual competitor research report with products, target customers,
    positioning, differentiators, evidence, and uncertainties.
    """,
    agent=research_agent
)

hierarchical_analysis_task = Task(
    description="""
    Analyze the competitor research.

    Identify:
    - Strengths
    - Weaknesses
    - Opportunities
    - Threats
    - Three actionable insights

    Connect major conclusions to evidence and avoid unsupported claims.
    """,
    expected_output="""
    A structured business analysis containing SWOT-style findings
    and three evidence-based actionable insights.
    """,
    agent=analysis_agent
)

hierarchical_marketing_task = Task(
    description="""
    Create a stakeholder-ready marketing recommendation based on the work
    performed by the specialist agents.

    Include:
    - Positioning angle
    - Target audience
    - Customer problem
    - Value proposition
    - Three marketing messages
    - Three practical actions
    - Risks and assumptions

    Review the specialist outputs before creating the recommendation.
    """,
    expected_output="""
    A stakeholder-ready marketing strategy containing positioning,
    audience, value proposition, messages, actions, and risks.
    """,
    agent=marketing_agent
)

hierarchical_crew = Crew(
    agents=[
        research_agent,
        analysis_agent,
        marketing_agent
    ],
    tasks=[
        hierarchical_research_task,
        hierarchical_analysis_task,
        hierarchical_marketing_task
    ],
    manager_agent=manager_agent,
    process=Process.hierarchical,
    verbose=True
)

print("Hierarchical Crew created.")
print("Process:", "hierarchical")
print("Manager:", manager_agent.role)
print("Number of specialist agents:", len(hierarchical_crew.agents))
print("Number of tasks:", len(hierarchical_crew.tasks))

print("\nStarting REAL hierarchical CrewAI execution...")

hierarchical_result = hierarchical_crew.kickoff()

print("\n===== HIERARCHICAL CREW RESULT =====")
print(hierarchical_result)

print("\n===== HIERARCHICAL CREW TASK OUTPUTS =====")

if hasattr(hierarchical_result, "tasks_output"):
    for index, task_output in enumerate(
        hierarchical_result.tasks_output,
        start=1
    ):
        print(f"\nTask {index} output:")
        print(task_output)

## Task 4 — Sequential vs Hierarchical Comparison

The comparison below records observations from the actual runs rather than inventing quality or latency results.
The final quality, execution time, and token usage should be populated from the real CrewAI executions.

In [ ]:
# Task 4 — Compare actual sequential and hierarchical runs

import time

def get_usage_info(result):
    usage = getattr(result, "token_usage", None)

    if usage is None:
        return {
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": None,
            "total_cost": None
        }

    return {
        "prompt_tokens": getattr(usage, "prompt_tokens", None),
        "completion_tokens": getattr(usage, "completion_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
        "total_cost": getattr(usage, "total_cost", None)
    }


sequential_usage = get_usage_info(sequential_result)
hierarchical_usage = get_usage_info(hierarchical_result)

print("Sequential usage:")
for key, value in sequential_usage.items():
    print(key, ":", value)

print("\nHierarchical usage:")
for key, value in hierarchical_usage.items():
    print(key, ":", value)

print("\nComparison:")
print("""
Sequential:
Pros:
- Simple execution flow
- Predictable task order
- Easier to debug
- Lower coordination complexity

Cons:
- Later work depends on earlier task completion
- Less flexible delegation
- A weak early result can affect later tasks

Hierarchical:
Pros:
- Manager can coordinate specialist work
- More flexible delegation
- Manager can review and coordinate outputs

Cons:
- More LLM calls and coordination overhead may occur
- Potentially higher latency and cost
- More complex debugging
- Manager behavior can introduce variability

When to use sequential:
Use sequential processing when the workflow has clear dependencies
and each task naturally follows the previous task.

When to use hierarchical:
Use hierarchical processing when the problem benefits from dynamic delegation,
review, coordination, or manager-level decision making.
""")

## Task 5 — Evaluation & Cost Awareness

The crew is evaluated using three simple criteria: factual grounding, completeness, and stakeholder-ready tone.
Token usage and cost are taken from the actual CrewAI result when available rather than being manually fabricated.

In [ ]:
# Task 5 — Evaluation & Cost Awareness

import pandas as pd

success_criteria = {
    "Factual grounding": (
        "Major claims are supported by research evidence and the agents avoid "
        "inventing competitor information."
    ),
    "Completeness": (
        "The final recommendation covers positioning, audience, problem, "
        "value proposition, messages, actions, and risks."
    ),
    "Stakeholder-ready tone": (
        "The final recommendation is clear, concise, professional, and actionable."
    )
}

print("Success criteria:")

for criterion, definition in success_criteria.items():
    print(f"\n{criterion}:")
    print(definition)


def extract_final_text(result):
    return str(result)


def automatic_structure_check(result):
    text = extract_final_text(result).lower()

    required_sections = [
        "position",
        "target",
        "value",
        "message",
        "action",
        "risk"
    ]

    found = sum(section in text for section in required_sections)

    if found >= 5:
        completeness_score = 5
    elif found >= 4:
        completeness_score = 4
    elif found >= 3:
        completeness_score = 3
    elif found >= 2:
        completeness_score = 2
    else:
        completeness_score = 1

    return completeness_score


sequential_completeness = automatic_structure_check(sequential_result)
hierarchical_completeness = automatic_structure_check(hierarchical_result)

print("\nActual structure checks:")
print("Sequential completeness score:", sequential_completeness, "/ 5")
print("Hierarchical completeness score:", hierarchical_completeness, "/ 5")


usage_table = pd.DataFrame([
    {
        "Run": "Sequential",
        "Prompt Tokens": sequential_usage["prompt_tokens"],
        "Completion Tokens": sequential_usage["completion_tokens"],
        "Total Tokens": sequential_usage["total_tokens"],
        "Approx Cost": sequential_usage["total_cost"]
    },
    {
        "Run": "Hierarchical",
        "Prompt Tokens": hierarchical_usage["prompt_tokens"],
        "Completion Tokens": hierarchical_usage["completion_tokens"],
        "Total Tokens": hierarchical_usage["total_tokens"],
        "Approx Cost": hierarchical_usage["total_cost"]
    }
])

print("\n===== ACTUAL TOKEN / COST INFORMATION =====")
print(usage_table.to_string(index=False))


print("\n===== MANUAL EVALUATION RUBRIC =====")

evaluation_rubric = pd.DataFrame([
    {
        "Criterion": "Factual grounding",
        "Score range": "1-5",
        "1": "Mostly unsupported claims",
        "3": "Mixed evidence quality",
        "5": "Claims consistently grounded in research"
    },
    {
        "Criterion": "Completeness",
        "Score range": "1-5",
        "1": "Major required sections missing",
        "3": "Most required sections present",
        "5": "All required sections are covered"
    },
    {
        "Criterion": "Stakeholder-ready tone",
        "Score range": "1-5",
        "1": "Unclear or informal",
        "3": "Generally understandable",
        "5": "Clear, professional, and actionable"
    }
])

print(evaluation_rubric.to_string(index=False))


print("\n===== THREE-RUN EVALUATION TEMPLATE =====")

three_run_scores = pd.DataFrame([
    {
        "Run": "Run 1",
        "Factual grounding (1-5)": "",
        "Completeness (1-5)": "",
        "Stakeholder tone (1-5)": "",
        "Total (out of 15)": ""
    },
    {
        "Run": "Run 2",
        "Factual grounding (1-5)": "",
        "Completeness (1-5)": "",
        "Stakeholder tone (1-5)": "",
        "Total (out of 15)": ""
    },
    {
        "Run": "Run 3",
        "Factual grounding (1-5)": "",
        "Completeness (1-5)": "",
        "Stakeholder tone (1-5)": "",
        "Total (out of 15)": ""
    }
])

print(three_run_scores.to_string(index=False))


print("\n===== FINAL MODEL COMPARISON =====")

comparison = pd.DataFrame([
    {
        "Approach": "Single Agent",
        "Pros": "Simple, low coordination overhead",
        "Cons": "Less specialization",
        "Best Use": "Small and straightforward tasks"
    },
    {
        "Approach": "Sequential Crew",
        "Pros": "Clear specialization and predictable workflow",
        "Cons": "Fixed task order and dependency chain",
        "Best Use": "Multi-step workflows with clear dependencies"
    },
    {
        "Approach": "Hierarchical Crew",
        "Pros": "Dynamic delegation and manager review",
        "Cons": "Higher complexity and potentially higher cost",
        "Best Use": "Complex workflows requiring coordination"
    }
])

print(comparison.to_string(index=False))

## Final Conclusion

This CrewAI project demonstrates a real multi-agent workflow in which specialized agents perform research, business analysis, and marketing strategy.
The sequential approach provides a predictable and easy-to-debug workflow, while the hierarchical approach adds manager-based delegation and review.
The actual token usage, execution behavior, and output quality should be considered when deciding whether the additional coordination is worthwhile.
For this business task, a multi-agent crew is most valuable when the work requires genuinely different specialist responsibilities; a single agent may remain preferable for simpler tasks because it reduces cost and complexity.